In [ ]:
from esm.sdk.forge import ESM3ForgeInferenceClient
from esm.sdk.api import ESMProtein, LogitsConfig
import pickle
import pandas as pd
import numpy as np

protein = ESMProtein(sequence="AAAAA")
forge_client = ESM3ForgeInferenceClient(model="esmc-6b-2024-12", url="https://forge.evolutionaryscale.ai", token="xxxxx")
protein_tensor = forge_client.encode(protein)
logits_output = forge_client.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.embeddings)

tensor([[[ 0.2266, -0.5039,  0.4531,  ...,  0.0732, -0.4180, -0.0056],
         [-0.0972, -0.5000,  0.4961,  ..., -0.6836,  0.3633, -0.6641],
         [ 0.6484,  0.1973,  0.2500,  ..., -0.3242, -0.2910,  1.2969],
         ...,
         [-0.3809, -0.0977, -0.0188,  ...,  0.2090, -0.1621,  0.0913],
         [ 1.3125,  0.2812,  1.4062,  ...,  0.8438,  0.4824,  0.7305],
         [ 1.1797,  0.5547,  0.3438,  ...,  1.4062, -0.4609, -0.8633]]],
       dtype=torch.bfloat16)


In [2]:
logits_output.embeddings.shape

torch.Size([1, 7, 2560])

In [5]:
from Bio import SeqIO
import torch
import time
import pickle
import os
from esm.sdk.forge import ESM3ForgeInferenceClient
from esm.sdk.api import ESMProtein, LogitsConfig
import requests.exceptions

# 读取氨基酸序列
seqList = []  
idList = []

for seq_record in SeqIO.parse('./excel/uncertain200.fasta', "fasta"):
    seq = str(seq_record.seq).upper()
    id = str(seq_record.id)
    
    seqList.append(str(seq))
    idList.append(str(id))

embedds, labels = [], []
neg_dic = {"ID": idList, "sequence": seqList, "embedd": [], "label": []}

# 创建临时保存目录
temp_dir = "./tempuncertain_embeddings"
os.makedirs(temp_dir, exist_ok=True)

# 检查是否有已经处理过的序列
processed_indices = []
for i in range(len(seqList)):
    temp_file = f"{temp_dir}/seq_{i}.pkl"
    if os.path.exists(temp_file):
        with open(temp_file, 'rb') as f:
            embedding = pickle.load(f)
            embedds.append(embedding)
            labels.append(1)
            processed_indices.append(i)
            print(f"已从缓存加载序列 {i+1}/{len(seqList)}: {idList[i]}")

remaining_indices = [i for i in range(len(seqList)) if i not in processed_indices]
print(f"开始处理剩余的 {len(remaining_indices)} 个氨基酸序列...")

# 初始化ESM客户端
max_retries = 5
retry_delay = 10  # 秒

for idx in remaining_indices:
    seq1 = seqList[idx]
    seq_id = idList[idx]
    print(f"处理序列 {idx+1}/{len(seqList)}: {seq_id}")
    
    for attempt in range(max_retries):
        try:
            # 每次尝试都重新初始化客户端
            forge_client = ESM3ForgeInferenceClient(
                model="esmc-6b-2024-12", 
                url="https://forge.evolutionaryscale.ai", 
                token="7crq2imFvwrXFA673YOSYO"
            )
            
            # 使用ESM模型提取特征
            protein = ESMProtein(sequence=seq1)
            protein_tensor = forge_client.encode(protein)
            logits_output = forge_client.logits(
                protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
            )
            
            # 获取embedding并去掉首尾两个标识符
            embedding = logits_output.embeddings
            embedding = embedding[:, 1:-1, :]  # 去掉首尾两个标识符
            
            # 转换为标准PyTorch张量并存储
            embedding = embedding.detach().clone()  # 安全地复制张量
            
            # 保存临时文件
            temp_file = f"{temp_dir}/seq_{idx}.pkl"
            with open(temp_file, 'wb') as f:
                pickle.dump(embedding, f)
            
            embedds.append(embedding)
            labels.append(1)
            
            print(f"成功处理序列 {idx+1}/{len(seqList)}")
            break  # 成功，跳出重试循环
            
        except Exception as e:
            print(f"处理序列 {idx+1}/{len(seqList)} 时出错: {e}")
            print(f"尝试 {attempt+1}/{max_retries} 失败")
            
            if attempt < max_retries - 1:
                print(f"等待 {retry_delay} 秒后重试...")
                time.sleep(retry_delay)
                retry_delay *= 1.5  # 指数退避策略
            else:
                print(f"达到最大重试次数，跳过序列 {idx+1}")
                # 在这里可以添加一个占位符或标记，表示这个序列处理失败
                embedds.append(None)  # 或者使用其他标记
                labels.append(1)
    
    # 每处理10个序列或最后一个序列，保存一次完整结果
    if (idx+1) % 10 == 0 or idx == remaining_indices[-1]:
        print(f"保存中间结果...")
        temp_neg_dic = {"ID": idList, "sequence": seqList, "embedd": embedds, "label": labels}
        with open('uncertain72_embedding_temp.pkl', 'wb') as f:
            pickle.dump(temp_neg_dic, f)
        print(f"已完成 {len(embedds)}/{len(seqList)} 个序列的特征提取")

print("所有序列的特征提取完成！")
neg_dic = {"ID": idList, "sequence": seqList, "embedd": embedds, "label": labels}

已从缓存加载序列 1/200: sp|O00187|MASP2_HUMAN
已从缓存加载序列 2/200: sp|O00217|NDUS8_HUMAN
已从缓存加载序列 3/200: sp|O00754|MA2B1_HUMAN
已从缓存加载序列 4/200: sp|O14521|DHSD_HUMAN
已从缓存加载序列 5/200: sp|O14618|CCS_HUMAN
已从缓存加载序列 6/200: sp|O14745|NHRF1_HUMAN
已从缓存加载序列 7/200: sp|O14746|TERT_HUMAN
已从缓存加载序列 8/200: sp|O14773|TPP1_HUMAN
已从缓存加载序列 9/200: sp|O14802|RPC1_HUMAN
已从缓存加载序列 10/200: sp|O15305|PMM2_HUMAN
已从缓存加载序列 11/200: sp|O43520|AT8B1_HUMAN
已从缓存加载序列 12/200: sp|O43766|LIAS_HUMAN
已从缓存加载序列 13/200: sp|O43918|AIRE_HUMAN
已从缓存加载序列 14/200: sp|O60260|PRKN_HUMAN
已从缓存加载序列 15/200: sp|O60566|BUB1B_HUMAN
已从缓存加载序列 16/200: sp|O75398|DEAF1_HUMAN
已从缓存加载序列 17/200: sp|O75746|S2512_HUMAN
已从缓存加载序列 18/200: sp|O75792|RNH2A_HUMAN
已从缓存加载序列 19/200: sp|O75923|DYSF_HUMAN
已从缓存加载序列 20/200: sp|O95180|CAC1H_HUMAN
已从缓存加载序列 21/200: sp|O95243|MBD4_HUMAN
已从缓存加载序列 22/200: sp|O95450|ATS2_HUMAN
已从缓存加载序列 23/200: sp|O95932|TGM3L_HUMAN
已从缓存加载序列 24/200: sp|O96017|CHK2_HUMAN
已从缓存加载序列 25/200: sp|P00439|PH4H_HUMAN
已从缓存加载序列 26/200: sp|P00533|EGFR_HUMAN
已从缓存加载序列 27

In [5]:
embedds[20].shape

torch.Size([1, 508, 2560])

In [ ]:
f_save = open('benign338_embedding.pkl', 'wb')
pickle.dump(neg_dic, f_save)
f_save.close()